🛰️ MintPy Daily-Use Code Repository — OpenSARLab Edition
Kernel Requirement: Every cell in this notebook must use the osl_mintpy kernel.
Check and switch it in the top-right corner of your Jupyter interface.
The default Python 3 kernel will fail with ModuleNotFoundError.

Universal Path Setup — Run This First in Every Session

**What it does:** Declares a master `DATA_DIR` variable pointing to your MintPy output folder.  
All 18 snippets below depend on this variable. Run this cell first, every time.

In [1]:
import os

# Change this to your actual MintPy geo-coded output folder in OpenSARLab.
# Typical OpenSARLab paths begin with /home/jovyan/
DATA_DIR = "/home/jovyan/dhaka/HyP3_downloads/MintPy"

# Verify the path actually exists before proceeding
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}\nCheck your path and try again.")

print(f"✅ Data directory confirmed: {DATA_DIR}")

✅ Data directory confirmed: /home/jovyan/dhaka/HyP3_downloads/MintPy


18 — Save a Filtered/Processed NumPy Array Back to HDF5

**What it does:** Writes a NumPy array (e.g. your filtered or decomposed velocity map)
back to disk as an HDF5 file, preserving the original spatial metadata so the result
remains geo-referenced and readable by MintPy tools.  
**When to use:** After generating a custom product (masked velocity, vertical component,
anomaly map) that you want to persist, share, or load in a later session without
rerunning the computation.


In [2]:
import h5py
import numpy as np
from mintpy.utils import readfile

# --- Input: load original velocity + metadata ---
vel_path = f"{DATA_DIR}/velocity.h5"
velocity, meta = readfile.read(vel_path)

# --- Apply your processing (example: coherence mask + vertical decomposition) ---
coh_path = f"{DATA_DIR}/temporalCoherence.h5"
coherence, _ = readfile.read(coh_path)
inc_angle_rad = np.deg2rad(float(meta.get('INCIDENCE_ANGLE', 33.0)))

processed = np.where(coherence >= 0.70, velocity, np.nan)
processed = processed / np.cos(inc_angle_rad)   # LOS → vertical

# --- Save back to HDF5 ---
output_path = f"{DATA_DIR}/velocity_vertical_masked.h5"

with h5py.File(output_path, 'w') as f:
    ds = f.create_dataset('velocity', data=processed.astype(np.float32), compression='gzip')
    # Write all original metadata attributes onto the dataset
    for key, value in meta.items():
        try:
            ds.attrs[key] = value
        except Exception:
            pass   # skip any non-serializable attribute silently

print(f"✅ Processed array saved to: {output_path}")
print(f"   Shape  : {processed.shape}")
print(f"   Valid px: {np.sum(~np.isnan(processed)):,}")

✅ Processed array saved to: /home/jovyan/dhaka/HyP3_downloads/MintPy/velocity_vertical_masked.h5
   Shape  : (314, 249)
   Valid px: 7,913
